# Task 4: Forecasting Access and Usage (2025-2027)

Predict Account Ownership and Digital Payment Usage with uncertainty intervals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from datetime import datetime

# Load enriched data
df = pd.read_csv('../data/enriched/ethiopia_fi_unified_data_enriched.csv')
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
obs = df[df['record_type'] == 'observation']

## Define Targets & Prepare Data

In [ ]:
# Targets: Access and Usage
targets = ['ACC_OWNERSHIP', 'USG_DIGITAL_PAYMENT']

# Create a dataframe with year as numeric feature
def prepare_target_data(indicator_code):
    sub = obs[obs['indicator_code'] == indicator_code].copy()
    sub['year'] = sub['observation_date'].dt.year
    sub = sub.sort_values('year')
    return sub[['year', 'value_numeric']].dropna()

acc_data = prepare_target_data('ACC_OWNERSHIP')
usage_data = prepare_target_data('USG_DIGITAL_PAYMENT')

print("Access data:")
print(acc_data)
print("\nUsage data:")
print(usage_data)

## Fit Trend Models & Generate Baseline Forecasts

In [ ]:
def forecast_with_ols(data, years_forecast=[2025, 2026, 2027]):
    X = data['year'].values.reshape(-1, 1)
    y = data['value_numeric'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    # Predict historical
    y_pred = model.predict(X)
    resid = y - y_pred
    sigma = np.std(resid)
    
    # Forecast future
    X_future = np.array(years_forecast).reshape(-1, 1)
    y_future = model.predict(X_future)
    
    # 95% Prediction Interval (approximate)
    lower = y_future - 1.96 * sigma
    upper = y_future + 1.96 * sigma
    
    results = pd.DataFrame({
        'year': years_forecast,
        'forecast': y_future,
        'lower_ci': lower,
        'upper_ci': upper
    })
    
    return results, model, sigma

acc_forecast, acc_model, acc_sigma = forecast_with_ols(acc_data)
usage_forecast, usage_model, usage_sigma = forecast_with_ols(usage_data)

print("Access Forecast (Baseline):")
print(acc_forecast)
print("\nUsage Forecast (Baseline):")
print(usage_forecast)

## Scenario Analysis (Incorporating Events)

We apply event impacts from the association matrix (Task 3).
Scenario: Base = Baseline + estimated event effects.

In [ ]:
# Define event impacts for 2025-2027
# From the association matrix: EVT_INTEROP_MANDATE adds +2.5 pp to USG_DIGITAL_PAYMENT with 12-month lag.
# So effect kicks in mid-2026.

def apply_scenario(forecast_df, indicator, event_impacts):
    df_scenario = forecast_df.copy()
    for year in df_scenario['year']:
        impact = 0
        for event, effect_year, magnitude in event_impacts:
            if year >= effect_year:
                impact += magnitude
        df_scenario.loc[df_scenario['year'] == year, 'forecast'] += impact
        df_scenario.loc[df_scenario['year'] == year, 'lower_ci'] += impact
        df_scenario.loc[df_scenario['year'] == year, 'upper_ci'] += impact
    return df_scenario

# Define events: (event_name, start_year, magnitude_pp)
access_events = []  # No direct access events in our enriched data yet
usage_events = [('Interoperability Mandate', 2026, 2.5)]  # lagged effect

acc_scenario = apply_scenario(acc_forecast, 'ACC_OWNERSHIP', access_events)
usage_scenario = apply_scenario(usage_forecast, 'USG_DIGITAL_PAYMENT', usage_events)

print("Access Scenario:")
print(acc_scenario)
print("\nUsage Scenario:")
print(usage_scenario)

## Visualise Forecasts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Access Plot
ax1 = axes[0]
ax1.plot(acc_data['year'], acc_data['value_numeric'], 'o-', label='Historical', color='blue')
ax1.plot(acc_forecast['year'], acc_forecast['forecast'], 's--', label='Baseline', color='darkblue')
ax1.fill_between(acc_forecast['year'], acc_forecast['lower_ci'], acc_forecast['upper_ci'], alpha=0.2, color='blue')
ax1.set_title('Access (Account Ownership)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Percentage of Adults')
ax1.legend()
ax1.grid(True)

# Usage Plot
ax2 = axes[1]
ax2.plot(usage_data['year'], usage_data['value_numeric'], 'o-', label='Historical', color='green')
ax2.plot(usage_forecast['year'], usage_forecast['forecast'], 's--', label='Baseline', color='darkgreen')
ax2.plot(usage_scenario['year'], usage_scenario['forecast'], '^--', label='With Events', color='orange')
ax2.fill_between(usage_forecast['year'], usage_forecast['lower_ci'], usage_forecast['upper_ci'], alpha=0.2, color='green')
ax2.set_title('Usage (Digital Payment Adoption)')
ax2.set_xlabel('Year')
ax2.set_ylabel('Percentage of Adults')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('../forecast_plot.png', dpi=150)
plt.show()

## Interpretation

- Access: Forecast shows slow growth (reaching ~51% by 2027) due to the flattening trend.
- Usage: Baseline reaches ~40% by 2027. The interoperability mandate pushes this to ~42.5% (optimistic).
- Uncertainty: Wide confidence intervals (±3-5 pp) reflect sparse data.
- Key Drivers: The largest impact comes from digital infrastructure and policy that directly enable payments, rather than broad account opening.
- Policy Insight: To accelerate access, targeted interventions (e.g., gender-focused, rural agent networks) are needed beyond what the current events capture.